# 1주차 4교시 학생용 복습 완성본

이 파일은 수업이 모두 끝난 뒤에 받는 복습용 자료입니다. 학생용 실습 노트북에서 먼저 직접 풀고, 그다음 이 파일의 완성 코드와 비교합니다.

코드를 그대로 다시 실행하는 데서 끝내지 말고, 각 Q에서 바뀐 핵심 한 줄과 그 이유를 말해 보세요.

# 1주차 4교시 — Pandas로 CSV를 읽고 상태 점검하기

## 학생용 복습 완성본

**수업이 끝나면 할 수 있는 것:** 회원 CSV를 DataFrame으로 읽고 행·열·자료형·결측치·선택 결과를 설명합니다.

> 수업에서 먼저 푼 뒤 정답 코드와 비교하세요. 결과만 보지 말고 어떤 빈칸이 왜 바뀌었는지 한 문장으로 다시 설명합니다.


## 50분 운영 리듬

- 0~5분 · 배열과 표 차이
- 5~13분 · read_csv
- 13~24분 · head·shape·columns
- 24~33분 · info·dtypes·isna
- 33~42분 · Series·DataFrame·loc·iloc
- 42~47분 · 오류 복구
- 47~50분 · 새 파일 상태 점검


## 준비 셀 — 데이터와 라이브러리 연결

- **뜻:** 오늘 사용할 도구와 데이터 폴더를 준비합니다.
- **이유:** 뒤의 Q가 데이터 위치와 앞 교시에 의존하지 않게 만듭니다.
- **정상:** 데이터 폴더 경로가 출력됩니다.
- **복구:** 프로젝트 폴더 안에서 노트북을 열고 이 셀부터 다시 실행합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_week1_data_dir():
    """어디에서 노트북을 열어도 수업 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "01_week1",
            base / "matchmaking_data" / "01_week1",
            base / "01_week1",
        ]
        for candidate in candidates:
            if (candidate / "matchmaking_members.csv").exists():
                return candidate
    raise FileNotFoundError("matchmaking_members.csv를 찾지 못했습니다. 프로젝트 폴더 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week1_data_dir()
print("데이터 폴더:", DATA_DIR)


## Q1. CSV를 읽기 전에 경로와 표의 크기를 예측하세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: read_csv는 쉼표 파일을 행·열이 있는 DataFrame으로 바꿉니다.
> 
> 이유: 파일을 읽어야 열 이름 기반의 Pandas 기능을 사용할 수 있습니다.
> 
> 정상: 데이터 폴더가 출력되고 members shape는 (1003, 21)입니다.
> 
> 복구: FileNotFoundError면 Path.cwd()와 DATA_DIR 출력부터 확인합니다.

### 실행 전에 생각할 질문

1000명이 아니라 1003행인 이유는 뒤 교시에서 찾을 의도적 중복 3행 때문입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
members_path = DATA_DIR / "matchmaking_members.csv"
prediction_shape = "(1003, 21)"
members = pd.read_csv(members_path)
print("예상:", prediction_shape)
print("실제:", members.shape)
assert members.shape == (1003, 21)


## Q2. head·shape·columns는 서로 다른 질문입니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: head는 처음 몇 행, shape는 크기, columns는 열 이름에 답합니다.
> 
> 이유: 계산 전에 파일이 맞게 읽혔는지 가장 싸게 확인하기 위해서입니다.
> 
> 정상: 첫 5행, 1003×21, member_id부터 meeting_scheduled까지 보입니다.
> 
> 복구: 출력이 너무 길면 head(3)과 columns.tolist()를 따로 실행합니다.

### 실행 전에 생각할 질문

‘몇 행인가?’에 head로 답할 수 있는지 토론하세요. 정답은 shape입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
print(members.head(3))
print("행·열:", members.shape)
print("열 이름:", members.columns.tolist())
assert "meeting_scheduled" in members.columns


## Q3. info·dtypes·isna는 표의 위험을 찾습니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: info는 전체 요약, dtypes는 열별 종류, isna().sum은 빈칸 수를 보여줍니다.
> 
> 이유: 연봉 열이 object이고 두 열에 결측치가 있음을 계산 전에 발견하기 위해서입니다.
> 
> 정상: 연봉 object, 연봉 6개·응답률 9개 NaN이 보입니다.
> 
> 복구: info()는 print로 감싸지 않아도 되며, isna 뒤에 괄호를 빠뜨리지 않습니다.

### 실행 전에 생각할 질문

연봉 열에 숫자가 많아 보이는데 왜 object인지 다음 교시의 정제 과제로 남깁니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
members.info()
print(members.dtypes[["age", "annual_income_10k_krw", "meeting_scheduled"]])
missing = members.isna().sum().sort_values(ascending=False)
print(missing.head())
assert missing["annual_income_10k_krw"] == 6
assert missing["recommendation_response_rate_prior"] == 9


## Q4. describe는 숫자 열에 ‘범위가 이상한가’를 묻습니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: 개수·평균·표준편차·최솟값·사분위·최댓값을 요약합니다.
> 
> 이유: 1000행을 직접 보지 않고 나이·활동 횟수 범위를 빠르게 점검하기 위해서입니다.
> 
> 정상: age의 count 1003과 min·max가 표시됩니다.
> 
> 복구: 숫자 열만 선택했는지, 열 이름 철자를 확인합니다.

### 실행 전에 생각할 질문

member_id 평균이 업무적으로 무의미한 이유를 말한 뒤 숫자 열도 질문에 맞춰 고릅니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
summary = members[["age", "profile_completion_rate", "profile_views_30d"]].describe()
print(summary)
assert int(summary.loc["count", "age"]) == 1003


## Q5. 한 열은 Series, 여러 열은 DataFrame입니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: members['age']는 1차원 Series, members[['age','region']]은 2차원 DataFrame입니다.
> 
> 이유: 다음 함수가 1차원과 2차원 중 무엇을 요구하는지 구별하기 위해서입니다.
> 
> 정상: Series와 DataFrame, (1003, 2)가 출력됩니다.
> 
> 복구: 여러 열은 안쪽에 열 이름 list가 있어 대괄호가 두 겹입니다.

### 실행 전에 생각할 질문

대괄호 한 겹과 두 겹의 결과 type을 실행 전에 예상하세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
one_column = members["age"]
two_columns = members[["age", "residence_region"]]
print(type(one_column), type(two_columns), two_columns.shape)
assert isinstance(one_column, pd.Series)
assert isinstance(two_columns, pd.DataFrame)


## Q6. loc는 이름·조건, iloc는 위치번호로 고릅니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: loc는 행 조건과 열 이름, iloc는 행·열 위치 번호를 사용합니다.
> 
> 이유: ‘활성 회원의 특정 열’과 ‘화면의 3~7번째 줄’ 같은 서로 다른 선택을 표현하기 위해서입니다.
> 
> 정상: loc에는 활성 회원 3열, iloc에는 3~7번째 행 4열이 나옵니다.
> 
> 복구: loc의 열은 이름 list, iloc의 끝 위치는 포함되지 않습니다.

### 실행 전에 생각할 질문

iloc[2:7]은 사람 말로 몇 번째부터 몇 번째 행인지 답하세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
active_preview = members.loc[members["is_active"] == True, ["member_id", "name", "login_week1"]].head(3)
position_preview = members.iloc[2:7, 0:4]
print(active_preview)
print(position_preview)
assert position_preview.shape == (5,4)


## Q7. 도움 없이 새 문제 풀기 — 새 회원 파일 상태 점검 보고서

> **처음 배우는 사람을 위한 안내**
>
> 성공 조건: 새 파일을 읽고 shape·첫 2행·열 이름·자료형·결측치 총합을 설명합니다.
> 
> 힌트 정책: 먼저 질문 이름만 → 막히면 head/shape/info/isna 다섯 단어만 줍니다.
> 
> 정상: new 파일은 100행×21열, 결측치 0입니다.
> 
> 복구: 한꺼번에 실행하지 말고 경로→read_csv→shape 순서로 처음 달라지는 곳을 찾습니다.

### 실행 전에 생각할 질문

노트북의 정답을 복사하지 말고 ‘파일이 맞나→크기는→열은→종류는→빈칸은’ 순서로 작성하세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
new_members = pd.read_csv(DATA_DIR / "matchmaking_members_new.csv")
print(new_members.shape)
print(new_members.head(2))
print(new_members.columns.tolist())
print(new_members.dtypes.head())
print("결측치 총합:", new_members.isna().sum().sum())
assert new_members.shape == (100,21)
assert new_members.isna().sum().sum() == 0


## 마무리 질문

1. 오늘 가장 자주 쓸 기능 하나와 그 이유는?
2. 오늘 기능을 쓰게 되는 데이터의 신호는?
3. 정상 결과를 무엇으로 확인했나?
4. 다음 교시로 넘겨야 할 결과는?
